# ENキーワード スクリーニング

EN トレンドデータを対象に、ピーク予測モデルに適したキーワードを絞り込む。

**分析フロー**
1. EN peak ≥ MIN_PEAK(10) のキーワードを抽出
2. 波形を確認し、予測モデルに適さないパターンを除外
   - 全期間フラット / 季節性 / ENピーク済み / 直近データ不足（mean_last52 < 5）
3. 最終予測対象キーワードを確定


In [1]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import logging
for _name in ['cmdstanpy', 'prophet', 'stan', 'pystan']:
    logging.getLogger(_name).setLevel(logging.ERROR)

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from prophet import Prophet
from trend_utils import load_category, get_peak, PERIODS, get_mean_last52, get_status
from screen_pairs import run_screening, MIN_PEAK, MIN_LAST52
from lag_estimator import AVG_LAG_MONTHS

FORECAST_WEEKS = 104


## Step 1: EN peak ≥ MIN_PEAK(10) でフィルタリング

monitoring ステータスのキーワードのうち EN ピーク値が MIN_PEAK(10) 以上のものを抽出する。
MIN_PEAK の根拠は `lag_pairs_analysis.md` セクション 6.2 参照。

In [2]:
df = run_screening()
monitoring_en = df[
    (df['status'] == 'monitoring') &
    (df['geo_type'] == 'en') &
    (df['peak'].notna()) &
    (df['peak'] >= MIN_PEAK)
]
step1 = monitoring_en.loc[monitoring_en.groupby('concept')['peak'].idxmax()]
step1[['concept', 'keyword', 'period', 'peak', 'peak_date', 'rolling_max']].reset_index(drop=True)
EXCLUDED_SO_FAR = set()


## Step 2: 波形確認・除外判断

EN peak ≥ MIN_PEAK でも、以下のパターンは予測対象から除外する。

- **除外①全期間フラット**: トレンドサイクルが存在しない定番アイテム
- **除外②季節性**: 毎年同時期に規則的にピークが来る（ラグモデル不適）
- **除外③ENピーク済み**: JP波及タイミングが既に過去
- **除外④直近データ不足**: mean_last52 < MIN_LAST52(5) → 一時スパイクのみでトレンド未形成


In [3]:
def plot_waves(keywords, title, ncols=2):
    n = len(keywords)
    cols = min(n, ncols)
    rows = (n + cols - 1) // cols
    fig = make_subplots(rows=rows, cols=cols,
        subplot_titles=[k[0] for k in keywords],
        vertical_spacing=0.15, horizontal_spacing=0.08)
    for i, (label, kw, cat, period) in enumerate(keywords):
        row, col = i // cols + 1, i % cols + 1
        s = load_category(cat, period)[kw]
        peak_val, peak_date = get_peak(s)
        fig.add_trace(go.Scatter(x=s.index, y=s.values,
            line=dict(color='#4472C4', width=2), showlegend=False), row=row, col=col)
        fig.add_trace(go.Scatter(x=[peak_date], y=[peak_val],
            mode='markers+text', marker=dict(color='#E15759', size=10),
            text=[f'<b>{peak_val}</b>'], textposition='top center',
            textfont=dict(size=11), showlegend=False), row=row, col=col)
    fig.update_layout(height=380*rows, template='plotly_white',
        title=dict(text=title, font=dict(size=14)))
    fig.update_yaxes(range=[0, 105], title_text='トレンドスコア')
    fig.show()

### 除外①: 全期間フラット（定番アイテム）

検索量が期間全体にわたって高水準で安定しており、トレンドサイクルが存在しない。
「波及」ではなく「もともと需要がある」アイテムのためラグ計算不可。

In [4]:
ex1 = ['crossbody bag', 'messenger bag']
EXCLUDED_SO_FAR.update(ex1)
plot_waves([
    ('crossbody bag (geo=US)', 'crossbody bag', 'items_en', '2021_2026'),
    ('messenger bag (geo=US)', 'messenger bag', 'items_en', '2021_2026'),
], '除外①: 全期間フラット（定番アイテム）', ncols=2)


### 除外②: 季節性アイテム（毎年同時期にピーク）

毎年秋冬に規則的にピークが来る。ラグモデルは「EN で一度爆発 → JP に波及」を前提としており、年次季節性には適用できない。
mid rise jeans はベースラインが右肩上がりだが、ピークは一貫して毎年11〜12月のため季節性と判断。

In [5]:
ex2 = ['military jacket', 'army jacket', 'sequin outfit', 'mid rise jeans']
EXCLUDED_SO_FAR.update(ex2)
plot_waves([
    ('military jacket (geo=US)', 'military jacket', 'items_en', '2021_2026'),
    ('army jacket (geo=US)',     'army jacket',     'items_en', '2021_2026'),
    ('sequin outfit (geo=US)',   'sequin outfit',   'items_en', '2021_2026'),
    ('mid rise jeans (geo=US)',  'mid rise jeans',  'items_en', '2021_2026'),
], '除外②: 季節性アイテム', ncols=2)


### 要監視: EN上昇初期（データ期間末端での急騰）

2021_2026 データの末端（2026-02）で急騰。それ以前は横ばいだが、EN で今まさに上昇し始めているサインの可能性がある。
直近データ不足（mean_last52 < MIN_LAST52）に該当するキーワードは除外④で除外される。


In [6]:
plot_waves([
    ('layered jewelry (geo=US)', 'layered jewelry', 'items_en', '2021_2026'),
], 'EN上昇初期（要監視）', ncols=1)


### 除外③: ENピーク済み（JP波及期間が過去）

EN ピークが古く、JP 波及タイミングが既に過去になるキーワード。
今から監視しても手遅れのため予測対象から除外する。


In [7]:
ex3 = ['mob wife aesthetic', 'old money aesthetic']
EXCLUDED_SO_FAR.update(ex3)
plot_waves([
    ('mob wife aesthetic (geo=US)',  'mob wife aesthetic',  'styles_en', '2021_2026'),
    ('old money aesthetic (geo=US)', 'old money aesthetic', 'styles_en', '2021_2026'),
], '除外③: ENピーク済み', ncols=2)


### 除外④: 直近データ不足（mean_last52 < MIN_LAST52）

EN peak ≥ MIN_PEAK を満たしていても、直近52週の平均検索量が低いキーワードは
一時的なスパイクのみでトレンドが定着していないと判断し除外する。

balletcore（LAG_PAIRS確定ペア）の peak ≈ 12 を踏まえ、`MIN_LAST52 = 5` を閾値として設定。


In [ ]:
TODAY    = pd.Timestamp.now().normalize()
DATA_END = load_category('styles_en', PERIODS[0]).index.max()

# ①②③で除外済みのキーワードを引いた残りでチェック
remaining = monitoring_en[~monitoring_en['keyword'].isin(EXCLUDED_SO_FAR)]
remaining = remaining.loc[remaining.groupby('keyword')['peak'].idxmax()]
remaining['mean_last52'] = remaining['keyword'].apply(get_mean_last52)

low_signal = remaining[remaining['mean_last52'] < MIN_LAST52]
ex4 = low_signal['keyword'].tolist()
EXCLUDED_SO_FAR.update(ex4)

print(f'除外④ 対象: {len(low_signal)}件')
kws_plot = []
for _, r in low_signal.iterrows():
    kw = r['keyword']
    for period in PERIODS:
        for cat in ['styles_en', 'items_en']:
            s = load_category(cat, period)
            if kw in s.columns:
                kws_plot.append((f"{kw} (mean52={r['mean_last52']})", kw, cat, period))
                break
        else:
            continue
        break

if kws_plot:
    plot_waves(kws_plot, f'除外④: 直近データ不足（mean_last52 < {MIN_LAST52}）', ncols=3)



## Step 3: 最終予測対象キーワード

Step 2 で除外したキーワードを除いた最終リストを確定する。
このリストを EN ピーク予測（）に引き渡す。


In [ ]:
monitoring_filtered = monitoring_en[~monitoring_en['keyword'].isin(EXCLUDED_SO_FAR)]
final_en = monitoring_filtered.loc[monitoring_filtered.groupby(['concept', 'keyword'])['peak'].idxmax()].copy()

jp_data = df[(df['status'] == 'monitoring') & (df['geo_type'].isin(['jp_ja', 'jp_en'])) & df['peak'].notna()]
jp_best = jp_data.loc[jp_data.groupby('concept')['peak'].idxmax()][['concept', 'keyword', 'peak', 'peak_date']].rename(
    columns={'keyword': 'jp_keyword', 'peak': 'jp_peak', 'peak_date': 'jp_peak_date'})

result = final_en.merge(jp_best, on='concept', how='left')

result['状態'] = result['peak_date'].apply(lambda x: get_status(x, DATA_END, TODAY))

result[['concept', 'keyword', 'peak', 'peak_date', '状態', 'jp_keyword', 'jp_peak', 'jp_peak_date']].rename(columns={
    'keyword':      'EN keyword',
    'peak':         'EN peak',
    'peak_date':    'EN peak date',
    'jp_keyword':   'JP keyword',
    'jp_peak':      'JP peak',
    'jp_peak_date': 'JP peak date',
}).reset_index(drop=True)

In [10]:
def load_jp_series_for_plot(jp_keyword):
    if pd.isna(jp_keyword):
        return None
    for period in PERIODS:
        for cat in ['styles_JP', 'items_JP', 'styles_en_jp', 'items_en_jp']:
            df_cat = load_category(cat, period)
            if jp_keyword in df_cat.columns:
                return df_cat[jp_keyword]
    return None

n    = len(result)
cols = 3
rows = (n + cols - 1) // cols
fig = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[f"{r['concept']} ({r['keyword']})" for _, r in result.iterrows()],
    vertical_spacing=0.15, horizontal_spacing=0.08,
)

for i, (_, row) in enumerate(result.iterrows()):
    r, c = i // cols + 1, i % cols + 1
    show_legend = i == 0

    # EN 系列
    en_series = None
    for period in PERIODS:
        for cat in ['styles_en', 'items_en']:
            s = load_category(cat, period)
            if row['keyword'] in s.columns:
                en_series = s[row['keyword']]
                break
        if en_series is not None:
            break

    if en_series is not None:
        en_peak_val, en_peak_date = get_peak(en_series)
        fig.add_trace(go.Scatter(
            x=en_series.index, y=en_series.values,
            name='EN (geo=US)', line=dict(color='#4472C4', width=2),
            legendgroup='en', showlegend=show_legend,
        ), row=r, col=c)
        fig.add_trace(go.Scatter(
            x=[en_peak_date], y=[en_peak_val],
            mode='markers+text', marker=dict(color='#4472C4', size=9),
            text=[f'<b>{en_peak_val}</b>'], textposition='top center', textfont=dict(size=10, color='#4472C4'),
            showlegend=False,
        ), row=r, col=c)

    # JP 系列（ピークマーカーなし）
    jp_series = load_jp_series_for_plot(row.get('jp_keyword'))
    if jp_series is not None:
        fig.add_trace(go.Scatter(
            x=jp_series.index, y=jp_series.values,
            name='JP (geo=JP)', line=dict(color='#E15759', width=2, dash='dot'),
            legendgroup='jp', showlegend=show_legend,
        ), row=r, col=c)

fig.update_layout(
    height=380 * rows, template='plotly_white',
    title=dict(text='最終予測対象キーワード EN/JP 波形', font=dict(size=14)),
    legend=dict(orientation='h', y=-0.05, x=0.5, xanchor='center'),
)
fig.update_yaxes(range=[0, 105], title_text='トレンドスコア')
fig.show()